In [ ]:
#Shape and types


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
airports = pd.read_csv(root / "data" / "raw" / "AA_AIRPORT_INFO.csv")
pax = pd.read_csv(root / "data" / "raw" / "AA_PAX_FORECASTING.csv")

print("Airports:", airports.shape)
print("Pax:", pax.shape)


In [ ]:
for name, df in [("AA_AIRPORT_INFO", airports), ("AA_PAX_FORECASTING", pax)]:
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print(df.dtypes, "\n")

In [ ]:
for name, df in [("AA_AIRPORT_INFO", airports), ("AA_PAX_FORECASTING", pax)]:
    print(name)
    print("columns:", list(df.columns))
    print("unique values per column:")
    print(df.nunique(dropna=False).to_string())
    print()


In [ ]:
#Column Name Meanings
#Airports:
#AIRPRT_CD: 3 LetterAirport code
#AIRPRT NM: Airport full name
#CITY_METRO_IATA_C: City metro IATA code, seperate from airport code
#STATE_PROVNC_CD: State or province code
#CNTRY_CD: Country code
#World Area Department of Transportation Code: 3 digit number that represents a geographic region
#LAT_DEGREE_QTY: Latitude of the airport in degrees
#LAT_MINUTE_QTY: Latitude of the airport in minutes 0-59
#LAT_SECOND_QTY: Latitude of the airport in seconds 0-59
#LAT_HEMSPHR_CD: Represents whether the latitude is in the Northern or Southern Hemisphere N or S
#LNGTD_DEGREE_QTY: Longitude of the airport in degrees
#LNGTD_MINUTE_QTY: Longitude of the airport in minutes 0-59
#LNGTD_SECOND_QTY: Longitude of the airport in seconds 0-59
#LNGTD_HEMSPHR_CD: Represents whether the longitude is in the Eastern or Western Hemisphere E or W
#LNGST_RUNWAY_FT_QTY: Length of the longest runway in feet
#ELEVATN_FT_QTY: Elevation of the airport in feet
#SF_LOAD_TMS: Snowflake Load Timestamp, represents the timestamp when the data was loaded into Snowflake

In [ ]:
#Column Name Meanings
#PAX
#DEP_MONTH: Month of departure
#DEP_YEAR: Year of departure
#ORG_AIRPRT: Origin airport code
#DES_AIRPRT: Destination airport code
#OD_PAX: Passengers from origin to destination in that departure month. This is the target.
#OD_REV_USD: Revenue from those same passengers in that same month. Not a feature.
#SF_LOAD_TMS: Snowflake load timestamp for this extract


In [ ]:
for name, df in [("airports", airports), ("pax", pax)]:
    miss = df.isna().mean().mul(100).round(2)
    print(f"{name} — % missing per column")
    print(miss[miss > 0].sort_values(ascending=False) if miss.any() else "No missing values", "\n")

In [ ]:
#Missingness
#The forecasting file has missing values in 0 columns.
#The airport file is missing CNTRY_CD on a few rows and STATE_PROVNC_CD on about half the rows.
#Those state blanks are non-US airports. Keep the column and fill the blank with MISSING.
#Do not drop it, and do not invent a state.


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt

target = "OD_PAX"
data = pax[target].dropna()
cap = data.quantile(0.99)   # value below which 99% of route-months fall

sns.set_style("whitegrid")
plt.figure(figsize=(10, 5))
sns.histplot(data[data <= cap], bins=50, kde=True)
plt.title(f"Distribution of {target} (bottom 99%, tail above {cap:,.0f} not shown)")
plt.xlabel("Passengers")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print(f"Median: {data.median():,.0f}  Mean: {data.mean():,.0f}  Max: {data.max():,.0f}  Skew: {data.skew():.2f}")

In [ ]:
#Distributions:
# OD_PAX is heavily right-skewed at 5.32.
# The median route-month has 503 passengers and the mean is 2765
# The max is 140194, so a small number of very busy routes pull the average way up



In [ ]:
print(pax.select_dtypes("number").corr()[target].sort_values(ascending=False).round(2))

In [ ]:
# Leakage
# OD_REV_USD correlates about 0.76 with OD_PAX, and about 0.95 after a log.
# It is the same month's revenue, so it is not a feature.
# Unnamed: 0 is the export row number. SF_LOAD_TMS is one load timestamp on every row.
# The airport table is static reference data, so joining it does not leak.
#
# Recommendation:
# Gradient-boosted trees.
# The target is right-skewed (skew 5.32), and a route-month table is not one time series.
# Tree splits are not thrown off by that scale the way a squared-error line is.
